# 🤖 Local GPT-4o Mini Automatic Labeling (Official 3K Candidates)

This notebook takes the **`candidates_official_3k.json`** file generated by your Kaggle run and automatically labels all 3,000 items with **GPT-4o Mini** (`openai/gpt-4o-mini`) using binary classification (`answers-it` vs `does-not-answer`).

### Key Features:
- **Checkpointing every 25 items**: If stopped or restarted, it resumes instantly with 0 lost progress and $0 re-billed.
- **Concurrent async requests**: Runs fast (~5 to 10 minutes total for 3,000 items).
- **Total Estimated Cost**: **~$0.28** for all 3,000 items.
- **Live Balance Diagnostics**: Checks the exact ratio of `answers-it` vs `does-not-answer` so you can decide the next move.


In [ ]:
# 1. Environment & API Key Verification
import os
import sys
import time
import json
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"OpenRouter API Key: {'✓ Present' if OPENROUTER_KEY else '✕ Missing (.env)'}")


In [ ]:
# 2. Load the 3,000 Official Candidates
DATA_DIR = Path("data")

OFFICIAL_CANDIDATES_FILE = DATA_DIR / "candidates_official_3k.json"
if not OFFICIAL_CANDIDATES_FILE.exists():
    # Check alternative paths
    for alt in [Path("candidates_official_3k.json"), Path("outputs/candidates_official_3k.json")]:
        if alt.exists():
            OFFICIAL_CANDIDATES_FILE = alt
            break

assert OFFICIAL_CANDIDATES_FILE.exists(), f"Could not find {OFFICIAL_CANDIDATES_FILE}. Please place the Kaggle downloaded candidates_official_3k.json in data/"

with open(OFFICIAL_CANDIDATES_FILE, "r", encoding="utf-8") as f:
    candidates = json.load(f)

print(f"✓ Loaded {len(candidates):,} official candidates across {len(set(c['scholar'] for c in candidates))} scholars.")
df_cands = pd.DataFrame(candidates)
print(df_cands['scholar'].value_counts().to_string())


In [ ]:
# 3. GPT-4o Mini Binary Labeling Engine with Auto-Checkpointing
LABELED_OUTPUT_FILE = DATA_DIR / "official_labeled_3k.json"

# Load existing checkpoints if resuming
labeled_cache = {}
if LABELED_OUTPUT_FILE.exists():
    try:
        with open(LABELED_OUTPUT_FILE, "r", encoding="utf-8") as f:
            labeled_cache = json.load(f)
        print(f"✓ Found existing checkpoint with {len(labeled_cache):,} items already labeled!")
    except Exception:
        pass

def save_checkpoint(cache: dict):
    with open(LABELED_OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

SYS_PROMPT = (
    "You are an expert Islamic fatwa relevance evaluation judge.\n"
    "Classify whether the provided Islamic fatwa directly and specifically answers the user's question.\n"
    "You must return ONLY ONE of the following two verdicts:\n"
    "1. 'answers-it': The fatwa directly and specifically answers the question asked.\n"
    "2. 'does-not-answer': The fatwa does NOT directly answer the question (including when it only touches on a related topic without directly answering, or is unrelated).\n\n"
    "Return a JSON object with:\n"
    "- 'verdict': exactly 'answers-it' or 'does-not-answer'\n"
    "- 'confidence': float between 0.0 and 1.0\n"
    "- 'explanation': brief explanation in Arabic."
)

def label_single_item(item: dict) -> dict:
    user_prompt = (
        f"User Question: {item['question']}\n\n"
        f"Scholar: {item['scholar']}\n"
        f"Title: {item['title']}\n"
        f"Fatwa Text: {item['answer_text'][:1400]}"
    )
    payload = {
        "model": "openai/gpt-4o-mini",
        "messages": [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.0,
        "response_format": {"type": "json_object"},
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"}
    
    t0 = time.perf_counter()
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    dt = time.perf_counter() - t0
    
    if resp.status_code >= 400:
        raise RuntimeError(f"API Error {resp.status_code}: {resp.text[:200]}")
        
    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {}) or {}
    
    verdict = "does-not-answer"
    conf = 1.0
    expl = ""
    try:
        m = re.search(r"\{.*?\}", content, re.DOTALL)
        if m:
            parsed = json.loads(m.group(0))
            raw_v = str(parsed.get("verdict", "")).lower().replace("_", "-").strip()
            verdict = "answers-it" if raw_v in ("answers-it", "answers", "yes", "answers_it") else "does-not-answer"
            conf = float(parsed.get("confidence", 1.0))
            expl = str(parsed.get("explanation", ""))
    except Exception:
        c_low = content.lower()
        verdict = "answers-it" if "answers-it" in c_low or "answers it" in c_low else "does-not-answer"
        
    return {
        "verdict": verdict,
        "confidence": conf,
        "explanation": expl,
        "cost": float(usage.get("cost", 0.0)),
        "latency_s": dt,
    }

# Filter items not yet labeled
pending_items = [it for it in candidates if str(it["id"]) not in labeled_cache]
print(f"Items to label: {len(pending_items):,} (Already labeled: {len(labeled_cache):,})")

# Run concurrent labeling (Workers = 8 for fast throughput)
MAX_WORKERS = 8
checkpoint_interval = 25
count_since_save = 0

if pending_items:
    pbar = tqdm(total=len(pending_items), desc="Labeling Official 3K with GPT-4o Mini")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_item = {executor.submit(label_single_item, it): it for it in pending_items}
        for future in as_completed(future_to_item):
            it = future_to_item[future]
            cid = str(it["id"])
            try:
                res = future.result()
                labeled_cache[cid] = {
                    **it,
                    "gpt_verdict": res["verdict"],
                    "gpt_confidence": res["confidence"],
                    "gpt_explanation": res["explanation"],
                    "cost": res["cost"],
                    "latency_s": res["latency_s"],
                }
                count_since_save += 1
                if count_since_save >= checkpoint_interval:
                    save_checkpoint(labeled_cache)
                    count_since_save = 0
            except Exception as e:
                print(f"\nError on item #{cid}: {e}")
            pbar.update(1)
    save_checkpoint(labeled_cache)
    print(f"\n✓ All {len(labeled_cache):,} items labeled and saved to {LABELED_OUTPUT_FILE}!")


In [ ]:
# 4. Balance Diagnostics & Analytics Dashboard
df_labeled = pd.DataFrame(list(labeled_cache.values()))

total_items = len(df_labeled)
answers_count = (df_labeled["gpt_verdict"] == "answers-it").sum()
no_answers_count = (df_labeled["gpt_verdict"] == "does-not-answer").sum()
total_cost = df_labeled["cost"].sum()

print("=" * 80)
print("OFFICIAL 3K DATASET BALANCE REPORT")
print("=" * 80)
print(f"Total items labeled:             {total_items:,}")
print(f"Answers it ('answers-it'):        {answers_count:,} ({100*answers_count/total_items:.1f}%)")
print(f"Does not answer ('does-not-answer'): {no_answers_count:,} ({100*no_answers_count/total_items:.1f}%)")
print(f"Total API cost for 3K:           ${total_cost:.4f}")
print("=" * 80)

print("\nVERDICT DISTRIBUTION BY SCHOLAR:")
scholar_summary = pd.crosstab(df_labeled["scholar"], df_labeled["gpt_verdict"], margins=True)
display(scholar_summary)

print("\nNEXT MOVE DECISION GUIDE:")
ratio = answers_count / total_items
if 0.45 <= ratio <= 0.55:
    print(f"✓ EXCELLENT BALANCE! Ratio is {ratio*100:.1f}%. The dataset is ready for Laya fine-tuning as-is.")
elif ratio > 0.55:
    diff = answers_count - no_answers_count
    print(f"⚠️ TOO MANY POSITIVES ({ratio*100:.1f}% answers-it). Recommendation: inject {diff:,} negative candidates from candidates_backup_3k.json to achieve a perfect 50/50 balance.")
else:
    print(f"⚠️ TOO MANY NEGATIVES ({ratio*100:.1f}% answers-it). Recommendation: add additional queries or adjust search mode.")
